In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

import numpy as np
import pandas as pd
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


# 32. OpenTelemetry LLM Observability with Plotly Dash

This Kaggle notebook uses `31-opentelemetry-llm-observability-gradio-e1.ipynb` as the functional baseline, keeps the same Grafana OTLP tracing and metrics setup, and swaps the UI layer to Plotly Dash.

Launch strategy in Kaggle:
- first try Dash notebook proxy mode with `jupyter_mode="tab"`
- if notebook proxy discovery fails, start Dash silently and expose it through an ngrok tunnel
- in both cases, the dashboard is intended to open in a separate browser tab


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import subprocess

print("=" * 70)
print("KAGGLE GPU CHECK")
print("=" * 70)
result = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
print(result.stdout)


In [ ]:
print("Installing llamatelemetry...")
!pip install -q --no-cache-dir --force-reinstall git+https://github.com/llamatelemetry/llamatelemetry.git@v0.1.1 > /dev/null
!pip install -q pyarrow pandas numpy scipy huggingface_hub

import llamatelemetry
print(f"llamatelemetry {llamatelemetry.__version__} installed")


In [ ]:
!pip install -q dash plotly pyngrok opentelemetry-api==1.37.0 opentelemetry-sdk==1.37.0 opentelemetry-proto==1.37.0 \
  opentelemetry-exporter-otlp-proto-common==1.37.0 opentelemetry-exporter-otlp-proto-http==1.37.0 \
  rich==13.9.4 --upgrade-strategy=only-if-needed


In [ ]:
# Step 1: Setup Secrets (Kaggle Secrets)
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()

GRAFANA_OTLP_ENDPOINT = secrets.get_secret("GRAFANA_OTLP_ENDPOINT").rstrip("/")
GRAFANA_OTLP_BASIC_B64 = secrets.get_secret("GRAFANA_OTLP_BASIC_B64")
GRAFANA_OTLP_INSTANCE_ID = secrets.get_secret("GRAFANA_OTLP_INSTANCE_ID")
GRAFANA_OTLP_TOKEN = secrets.get_secret("GRAFANA_OTLP_TOKEN")

HF_TOKEN = secrets.get_secret("HF_TOKEN")
login(HF_TOKEN)

try:
    NGROK_AUTHTOKEN = secrets.get_secret("NGROK_AUTHTOKEN")
except Exception:
    NGROK_AUTHTOKEN = None

os.environ["OTEL_EXPORTER_OTLP_PROTOCOL"] = "http/protobuf"
os.environ["OTEL_EXPORTER_OTLP_LOGS_ENDPOINT"] = f"{GRAFANA_OTLP_ENDPOINT}/v1/logs"
os.environ["OTEL_EXPORTER_OTLP_TRACES_ENDPOINT"] = f"{GRAFANA_OTLP_ENDPOINT}/v1/traces"
os.environ["OTEL_EXPORTER_OTLP_METRICS_ENDPOINT"] = f"{GRAFANA_OTLP_ENDPOINT}/v1/metrics"
os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = f"Authorization=Basic%20{GRAFANA_OTLP_BASIC_B64}"
os.environ["OTEL_TRACES_EXPORTER"] = "otlp"
os.environ["OTEL_METRICS_EXPORTER"] = "otlp"

print("Grafana OTLP endpoint configured:", GRAFANA_OTLP_ENDPOINT)
print("ngrok authtoken present:", bool(NGROK_AUTHTOKEN))


In [ ]:
# Step 2: OpenTelemetry Setup (Grafana OTLP + local memory export)
import logging
from opentelemetry import metrics, trace
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor, SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.sdk.metrics.export import PeriodicExportingMetricReader
from opentelemetry.exporter.otlp.proto.http.metric_exporter import OTLPMetricExporter

logging.getLogger().setLevel(logging.CRITICAL)
logging.getLogger("opentelemetry").setLevel(logging.CRITICAL)
logging.getLogger("opentelemetry").propagate = False

try:
    trace.get_tracer_provider().shutdown()
except Exception:
    pass

try:
    metrics.get_meter_provider().shutdown()
except Exception:
    pass

resource = Resource.create({
    "service.name": "llamatelemetry-dash-observability",
    "service.version": "0.1.1",
    "service.instance.id": "kaggle-dash-dashboard",
    "deployment.environment": "kaggle-notebook",
    "host.name": "kaggle-gpu-0",
    "gpu.model": "Tesla T4",
})

tracer_provider = TracerProvider(resource=resource)
span_exporter = OTLPSpanExporter(
    endpoint=f"{GRAFANA_OTLP_ENDPOINT}/v1/traces",
    headers={
        "Authorization": f"Basic {GRAFANA_OTLP_BASIC_B64}",
        "Content-Type": "application/x-protobuf",
    },
)
tracer_provider.add_span_processor(BatchSpanProcessor(span_exporter))
memory_exporter = InMemorySpanExporter()
tracer_provider.add_span_processor(SimpleSpanProcessor(memory_exporter))
trace.set_tracer_provider(tracer_provider)
tracer = trace.get_tracer(__name__)

metric_exporter = OTLPMetricExporter(
    endpoint=f"{GRAFANA_OTLP_ENDPOINT}/v1/metrics",
    headers={"Authorization": f"Basic {GRAFANA_OTLP_BASIC_B64}"},
)
metric_reader = PeriodicExportingMetricReader(metric_exporter, export_interval_millis=5000)
meter_provider = MeterProvider(resource=resource, metric_readers=[metric_reader])
metrics.set_meter_provider(meter_provider)
meter = metrics.get_meter(__name__)

request_counter = meter.create_counter(
    name="llm.requests.total",
    description="Total number of LLM requests",
    unit="1",
)
latency_histogram = meter.create_histogram(
    name="llm.request.duration",
    description="LLM request latency",
    unit="ms",
)
token_histogram = meter.create_histogram(
    name="llm.tokens.total",
    description="Token usage per request",
    unit="{token}",
)

with tracer.start_as_current_span("grafana.sanity") as span:
    span.set_attribute("check", "ok")

print("OpenTelemetry providers initialized for Grafana + local Dash dashboard")


In [ ]:
# Step 3: Download GGUF model and start llama.cpp server
import os
import torch
from huggingface_hub import hf_hub_download
from llamatelemetry.server import ServerManager

os.makedirs("/kaggle/working/models", exist_ok=True)

model_path = hf_hub_download(
    repo_id="bartowski/Qwen2.5-3B-Instruct-GGUF",
    filename="Qwen2.5-3B-Instruct-Q4_K_M.gguf",
    local_dir="/kaggle/working/models",
)
print(f"Model downloaded: {model_path}")

print(f"Detected {torch.cuda.device_count()} GPU(s)")
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

server = ServerManager(server_url="http://127.0.0.1:8080")
server.start_server(
    model_path=model_path,
    gpu_layers=99,
    tensor_split="1.0",
    port=8080,
    host="127.0.0.1",
    ctx_size=4096,
    batch_size=512,
)

print("llama.cpp server ready on http://127.0.0.1:8080")


In [ ]:
# Step 4: Instrumented inference helpers
import random
import pandas as pd
import time
from opentelemetry.trace import Status, StatusCode
from llamatelemetry.api import LlamaCppClient

class InstrumentedLLMClient:
    def __init__(self, base_url: str, tracer):
        self.client = LlamaCppClient(base_url)
        self.tracer = tracer

    def chat_completion(self, messages: list, **kwargs):
        model = kwargs.get("model", "unknown")
        max_tokens = kwargs.get("max_tokens", 100)
        temperature = kwargs.get("temperature", 0.7)

        with self.tracer.start_as_current_span(
            name=f"llm.chat.{model}",
            kind=trace.SpanKind.CLIENT,
        ) as span:
            try:
                span.set_attribute("llm.system", "llama.cpp")
                span.set_attribute("llm.model", model)
                span.set_attribute("llm.request.max_tokens", max_tokens)
                span.set_attribute("llm.request.temperature", temperature)
                span.set_attribute("llm.request.messages", len(messages))

                start_time = time.time()
                response = self.client.chat.completions.create(messages=messages, **kwargs)
                latency_ms = (time.time() - start_time) * 1000

                finish_reason = response.choices[0].finish_reason
                content = response.choices[0].message.content
                span.set_attribute("llm.response.finish_reason", finish_reason)
                span.set_attribute("llm.response.length", len(content))

                request_counter.add(
                    1,
                    attributes={"model": model, "finish_reason": finish_reason, "status": "success"},
                )
                latency_histogram.record(latency_ms, attributes={"model": model, "status": "success"})

                if hasattr(response, "usage"):
                    input_tokens = getattr(response.usage, "prompt_tokens", 0)
                    output_tokens = getattr(response.usage, "completion_tokens", 0)
                    span.set_attribute("llm.usage.input_tokens", input_tokens)
                    span.set_attribute("llm.usage.output_tokens", output_tokens)
                    token_histogram.record(input_tokens, attributes={"model": model, "token_type": "input"})
                    token_histogram.record(output_tokens, attributes={"model": model, "token_type": "output"})

                span.set_status(Status(StatusCode.OK))
                return response
            except Exception as exc:
                span.set_status(Status(StatusCode.ERROR, str(exc)))
                span.record_exception(exc)
                request_counter.add(
                    1,
                    attributes={
                        "model": model,
                        "status": "error",
                        "error_type": type(exc).__name__,
                    },
                )
                raise

llm = InstrumentedLLMClient("http://127.0.0.1:8080", tracer)

def generate_sample_requests(batch_size: int = 6):
    prompts = [
        "Explain transformer architecture in simple terms.",
        "What is GGUF and why is it useful for local inference?",
        "Describe KV cache behavior in llama.cpp.",
        "Why does quantization reduce memory usage?",
        "How does OpenTelemetry tracing help with LLM observability?",
        "Compare latency and throughput for batch inference.",
    ]

    outputs = []
    with tracer.start_as_current_span("llm.batch.requests"):
        for i in range(batch_size):
            prompt = prompts[i % len(prompts)]
            response = llm.chat_completion(
                messages=[{"role": "user", "content": prompt}],
                max_tokens=random.randint(64, 160),
                temperature=round(random.uniform(0.4, 0.9), 2),
            )
            outputs.append(
                {
                    "prompt": prompt,
                    "response": response.choices[0].message.content,
                    "finish_reason": response.choices[0].finish_reason,
                }
            )
            time.sleep(0.4)

    tracer_provider.force_flush()
    return pd.DataFrame(outputs)

sample_df = generate_sample_requests(batch_size=6)
sample_df.head()


In [ ]:
# Step 5: Prepare Dash dashboard state
import plotly.express as px
import plotly.graph_objects as go

def spans_to_dataframe():
    rows = []
    for span in memory_exporter.get_finished_spans():
        attrs = span.attributes or {}
        rows.append(
            {
                "span_name": span.name,
                "status": span.status.status_code.name,
                "start_time": pd.to_datetime(span.start_time, unit="ns"),
                "duration_ms": (span.end_time - span.start_time) / 1_000_000,
                "model": attrs.get("llm.model", "unknown"),
                "input_tokens": attrs.get("llm.usage.input_tokens", 0),
                "output_tokens": attrs.get("llm.usage.output_tokens", 0),
                "total_tokens": attrs.get("llm.usage.input_tokens", 0) + attrs.get("llm.usage.output_tokens", 0),
                "temperature": attrs.get("llm.request.temperature", None),
                "max_tokens": attrs.get("llm.request.max_tokens", None),
                "finish_reason": attrs.get("llm.response.finish_reason", "n/a"),
            }
        )

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values("start_time").reset_index(drop=True)
    return df

def empty_figure(title):
    return go.Figure().update_layout(title=title, template="plotly_white")

def build_dash_state():
    df = spans_to_dataframe()
    if df.empty:
        return {
            "summary": "No telemetry spans yet. Generate sample traffic from the dashboard.",
            "latency_fig": empty_figure("No spans captured yet"),
            "token_fig": empty_figure("No spans captured yet"),
            "request_data": [],
            "span_data": [],
        }

    recent_requests = sample_df.tail(10).copy() if 'sample_df' in globals() else pd.DataFrame(columns=["prompt", "response", "finish_reason"])
    latency_fig = px.histogram(
        df,
        x="duration_ms",
        nbins=20,
        color="status",
        title="Latency Distribution (ms)",
        template="plotly_white",
    )
    token_fig = px.line(
        df,
        x="start_time",
        y=["input_tokens", "output_tokens", "total_tokens"],
        title="Token Usage Over Time",
        markers=True,
        template="plotly_white",
    )

    summary = (
        f"Captured spans: {len(df)} | "
        f"Mean latency: {df['duration_ms'].mean():.2f} ms | "
        f"Total tokens: {int(df['total_tokens'].sum())} | "
        f"Models: {', '.join(sorted(df['model'].astype(str).unique()))} | "
        f"Grafana OTLP export: enabled"
    )

    span_table = df[["start_time", "span_name", "duration_ms", "model", "total_tokens", "status", "finish_reason"]].tail(25).copy()
    span_table["start_time"] = span_table["start_time"].astype(str)
    return {
        "summary": summary,
        "latency_fig": latency_fig,
        "token_fig": token_fig,
        "request_data": recent_requests.to_dict("records"),
        "span_data": span_table.to_dict("records"),
    }

build_dash_state()["summary"]


In [ ]:
# Step 6: Launch Dash in Kaggle
from IPython.display import HTML, display
from dash import Dash, dcc, html, Input, Output, State, callback_context, dash_table, jupyter_dash
from pyngrok import ngrok

app = Dash(__name__)
app.title = "LlamaTelemetry Dash Observability"

app.layout = html.Div(
    style={"fontFamily": "Arial, sans-serif", "padding": "20px", "backgroundColor": "#f8fafc"},
    children=[
        html.H1("LlamaTelemetry Dash Observability Dashboard"),
        html.P("Dash UI for local spans captured in Kaggle while traces and metrics are exported to Grafana OTLP."),
        html.Div(
            [
                html.Label("Sample request batch size"),
                dcc.Slider(id="batch-size", min=1, max=12, step=1, value=6),
                html.Br(),
                html.Button("Generate sample traffic", id="run-btn", n_clicks=0, style={"marginRight": "12px"}),
                html.Button("Refresh dashboard", id="refresh-btn", n_clicks=0),
            ],
            style={"marginBottom": "24px"},
        ),
        dcc.Markdown(id="summary", style={"padding": "12px", "background": "white", "border": "1px solid #dbe2ea"}),
        html.Div(
            [
                dcc.Graph(id="latency-graph", style={"width": "50%", "display": "inline-block"}),
                dcc.Graph(id="token-graph", style={"width": "50%", "display": "inline-block"}),
            ]
        ),
        html.H3("Recent sample requests"),
        dash_table.DataTable(
            id="request-table",
            columns=[
                {"name": "prompt", "id": "prompt"},
                {"name": "response", "id": "response"},
                {"name": "finish_reason", "id": "finish_reason"},
            ],
            page_size=10,
            style_table={"overflowX": "auto"},
            style_cell={"textAlign": "left", "whiteSpace": "normal", "height": "auto"},
        ),
        html.H3("Recent spans"),
        dash_table.DataTable(
            id="span-table",
            columns=[
                {"name": "start_time", "id": "start_time"},
                {"name": "span_name", "id": "span_name"},
                {"name": "duration_ms", "id": "duration_ms"},
                {"name": "model", "id": "model"},
                {"name": "total_tokens", "id": "total_tokens"},
                {"name": "status", "id": "status"},
                {"name": "finish_reason", "id": "finish_reason"},
            ],
            page_size=15,
            style_table={"overflowX": "auto"},
            style_cell={"textAlign": "left", "whiteSpace": "normal", "height": "auto"},
        ),
    ],
)

@app.callback(
    Output("summary", "children"),
    Output("latency-graph", "figure"),
    Output("token-graph", "figure"),
    Output("request-table", "data"),
    Output("span-table", "data"),
    Input("run-btn", "n_clicks"),
    Input("refresh-btn", "n_clicks"),
    State("batch-size", "value"),
)
def update_dashboard(run_clicks, refresh_clicks, batch_size):
    global sample_df
    ctx = callback_context
    if ctx.triggered and ctx.triggered[0]["prop_id"].startswith("run-btn"):
        sample_df = generate_sample_requests(batch_size=int(batch_size or 6))

    state = build_dash_state()
    return (
        state["summary"],
        state["latency_fig"],
        state["token_fig"],
        state["request_data"],
        state["span_data"],
    )

dash_port = 8050
proxy_mode_started = False
proxy_error = None
public_tunnel = None

try:
    jupyter_dash.infer_jupyter_proxy_config()
    print("Dash notebook proxy config inferred. Launching tab mode...")
    app.run(host="127.0.0.1", port=dash_port, jupyter_mode="tab")
    proxy_mode_started = True
except Exception as exc:
    proxy_error = exc
    print("Dash notebook proxy mode failed:", exc)

if not proxy_mode_started:
    if NGROK_AUTHTOKEN:
        ngrok.set_auth_token(NGROK_AUTHTOKEN)

    app.run(host="0.0.0.0", port=dash_port, jupyter_mode="_none")
    public_tunnel = ngrok.connect(addr=dash_port, proto="http", bind_tls=True)
    public_url = public_tunnel.public_url
    display(HTML(f'<a href="{public_url}" target="_blank">Open the Dash observability dashboard in a new tab</a>'))
    print("Dash tunnel URL:", public_url)
    if proxy_error is not None:
        print("Proxy mode fallback reason:", proxy_error)
